In [ ]:
# 9.6 Gradio Chatbot Interface
import gradio as gr

# Mock functions for standalone UI development
def make_initial_state():
    return {"stage": "idle", "clarify_group": None, "clarify_step": 0,
            "clarify_context": {}, "plan_type": None}

def chatbot_respond(user_input, state):
    l = user_input.lower()
    if "compression" in l or "sock" in l:
        return (
            "I found several similar categories in the **Compression Stockings** group.\n\n"
            "To narrow it down: **Is it footless (calf sleeves/tights) or regular stockings?**\n\n"
            "Options: Footless / Regular stockings"
        ), state
    if "vision" in l:
        return (
            "**Vision** (6 categories):\n\n"
            "- Contact Lenses\n- Eye Exam\n- Frames (prescription)\n"
            "- Lenses (prescription)\n- Safety Glasses (prescription)\n"
            "- Visual Training / Eye Therapy"
        ), state
    if "diabet" in l:
        return (
            "For **diabetes**, here are the relevant categories:\n\n"
            "1. **Blood glucose meter** (94%)\n"
            "2. **Glucose Monitoring Supplies/Sensors** (91%)\n"
            "3. **Glucose Monitoring Transmitter** (88%)\n"
            "4. **Insulin Pump** (79%)\n"
            "5. **Insulin Gun/Pen** (74%)"
        ), state
    if "brace" in l or "custom" in l:
        return (
            "**Brace** has 4 categories:\n\n"
            "- **Brace (Non-custom)** — off-the-shelf\n"
            "- **Brace, AFO/KAFO (Custom)**\n"
            "- **Brace, left knee (Custom)**\n"
            "- **Brace, right knee (Custom)**\n\n"
            "Key distinction: **custom vs off-the-shelf**, then type."
        ), state
    if "cpap" in l or "mask" in l:
        return (
            "The **CPAP** line has 3 components:\n\n"
            "- **CPAP, APAP, BIPAP Machine** — the device\n"
            "- **CPAP, APAP, BIPAP Mask** — the interface\n"
            "- **CPAP, APAP, BIPAP Supplies** — tubing, filters\n\n"
            "Since you mentioned a mask, I recommend:\n\n"
            "**CPAP, APAP, BIPAP Mask**\n"
            "Benefit: Medical Items — Respiratory Equipment\n"
            "Relevance: 96%"
        ), state
    if "massage" in l:
        return (
            "Yes, **Massage Therapy** is covered.\n\n"
            "**Massage Therapy**\n"
            "Benefit: Professional Services — Therapy\n"
            "Requires: Licensed Massage Therapist (RMT)\n"
            "Relevance: 97%"
        ), state
    return (
        "Based on your description, I recommend:\n\n"
        "**Physiotherapy** (95%)\n"
        "Benefit: Professional Services — Therapy\n"
        "Coverage: Combined therapy maximum"
    ), state

session_state = {"state": make_initial_state()}

def first_message(message, history):
    state = session_state["state"]
    response, state = chatbot_respond(message, state)
    session_state["state"] = state
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]
    return gr.update(visible=False), gr.update(visible=True), history, ""

def chat_message(message, history):
    state = session_state["state"]
    response, state = chatbot_respond(message, state)
    session_state["state"] = state
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": response}]
    return "", history

def first_example(example_text, history):
    return first_message(example_text, history)

def reset_chat():
    session_state["state"] = make_initial_state()
    return gr.update(visible=True), gr.update(visible=False), [], "", ""

custom_css = """
/* Base */
.gradio-container, .gradio-container * {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif !important;
}
footer { display: none !important; }

/* Welcome */
.welcome-wrap { text-align: center; padding: 36px 10px 12px; }
.welcome-wrap h2 {
    font-size: 32px; color: #1a8a52; margin-bottom: 6px;
    font-weight: 700; letter-spacing: -0.03em;
}
.welcome-wrap p { font-size: 16px; color: #171717; font-weight: 400; margin: 0; }

/* Hero input green border */
.hero-input {
    width: 100% !important;
    max-width: 680px !important;
    margin: 0 auto !important;
    border: 2px solid #1a8a52 !important;
    border-radius: 20px !important;
    background: #fff !important;
    box-shadow: 0 0 0 4px rgba(26,138,82,0.08) !important;
    padding: 4px 6px 4px 4px !important;
}
.hero-input:focus-within {
    box-shadow: 0 0 0 5px rgba(26,138,82,0.15) !important;
}

/* Kill Gradio inner borders */
.hero-input *,
.hero-input *:focus,
.hero-input *:focus-visible,
.hero-input *:focus-within,
.hero-input *:active {
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
    background: transparent !important;
}

.hero-input textarea {
    font-size: 16px !important;
    padding: 14px 16px !important;
    min-height: 64px !important;
    color: #171717 !important;
    line-height: 1.5 !important;
    border-radius: 16px !important;
}
.hero-input textarea::placeholder { color: #a3a3a3 !important; }

/* Send button (restore after nuke) */
.hero-input .send-welcome,
.hero-input .send-welcome button,
.hero-input button.send-welcome {
    border-radius: 12px !important;
    min-width: 44px !important; max-width: 44px !important;
    height: 44px !important; padding: 0 !important;
    background: #1a8a52 url("data:image/svg+xml,%3Csvg viewBox='0 0 24 24' fill='white' xmlns='http://www.w3.org/2000/svg'%3E%3Cpath d='M3.4 20.4l17.45-7.48a1 1 0 000-1.84L3.4 3.6a.993.993 0 00-1.39.91L2 9.12c0 .5.37.93.87.99L17 12 2.87 13.88c-.5.07-.87.5-.87 1l.01 4.61c0 .71.73 1.2 1.39.91z'/%3E%3C/svg%3E") center/18px no-repeat !important;
    color: transparent !important; cursor: pointer !important;
    flex-shrink: 0 !important;
}
.hero-input .send-welcome:hover,
.hero-input .send-welcome button:hover {
    background-color: #156f42 !important;
}

/* Welcome column layout */
.welcome-col { gap: 4px !important; }
.welcome-col > div { gap: 4px !important; }

/* Try asking */
.try-label {
    color: #737373; font-size: 13.5px; text-align: center;
    margin: 8px 0 2px; font-weight: 400;
}

/* Example chips */
.chips-wrap {
    display: flex !important; flex-wrap: wrap !important;
    justify-content: center !important; gap: 5px !important;
    max-width: 680px !important; margin: 0 auto !important;
    padding: 0 !important;
}
.chip {
    border: 1px solid #d4d4d8 !important;
    border-radius: 10px !important;
    background: none !important;
    font-size: 13.5px !important;
    padding: 8px 14px !important;
    color: #52525b !important;
    font-weight: 400 !important;
    line-height: 1.35 !important;
    min-width: auto !important;
    max-width: fit-content !important;
    width: auto !important;
    flex: none !important;
    box-shadow: none !important;
}
.chip:hover {
    background: #e8f5ef !important;
    border-color: #1a8a52 !important;
    color: #1a8a52 !important;
}
.chips-wrap > button, .chips-wrap .gr-button {
    flex-grow: 0 !important; flex-shrink: 0 !important;
}

/* Chat screen */
.chat-header span { font-size: 15px !important; font-weight: 600 !important; color: #1a8a52 !important; }
.new-chat-btn {
    background: #1a8a52 !important; color: #fff !important;
    border: none !important; border-radius: 8px !important;
    font-size: 13px !important; padding: 6px 16px !important;
    box-shadow: none !important;
}
.new-chat-btn:hover { background: #156f42 !important; }
.message { line-height: 1.65 !important; font-size: 14.5px !important; }

.chat-bar textarea {
    border: 1.5px solid #e5e5e5 !important; border-radius: 16px !important;
    font-size: 14.5px !important; padding: 12px 16px !important;
    color: #171717 !important; box-shadow: none !important;
}
.chat-bar textarea:focus {
    border-color: #1a8a52 !important;
    box-shadow: 0 0 0 3px rgba(26,138,82,0.08) !important;
}

.send-chat, .send-chat button {
    border-radius: 10px !important;
    min-width: 38px !important; max-width: 38px !important;
    height: 38px !important; padding: 0 !important;
    background: #1a8a52 url("data:image/svg+xml,%3Csvg viewBox='0 0 24 24' fill='white' xmlns='http://www.w3.org/2000/svg'%3E%3Cpath d='M3.4 20.4l17.45-7.48a1 1 0 000-1.84L3.4 3.6a.993.993 0 00-1.39.91L2 9.12c0 .5.37.93.87.99L17 12 2.87 13.88c-.5.07-.87.5-.87 1l.01 4.61c0 .71.73 1.2 1.39.91z'/%3E%3C/svg%3E") center/16px no-repeat !important;
    color: transparent !important; cursor: pointer !important;
    flex-shrink: 0 !important; border: none !important;
    box-shadow: none !important;
}
.send-chat:hover, .send-chat button:hover { background-color: #156f42 !important; }
"""

with gr.Blocks(title="GreenShield ClaimBot", css=custom_css, theme=gr.themes.Soft()) as demo:

    # Welcome
    with gr.Column(visible=True, elem_classes=["welcome-col"]) as welcome_section:
        gr.HTML(
            '<div class="welcome-wrap">'
            '<h2>GreenShield ClaimBot</h2>'
            '<p>Ask anything about your GreenShield+ insurance claim</p>'
            '</div>'
        )
        with gr.Group(elem_classes=["hero-input"]):
            with gr.Row():
                welcome_input = gr.Textbox(
                    placeholder="Ask about claim categories, coverage, benefits...",
                    show_label=False, container=False,
                    autofocus=True, scale=8, lines=2,
                )
                welcome_send = gr.Button("", variant="primary",
                                         elem_classes=["send-welcome"], scale=1, icon=None)

        gr.HTML('<div class="try-label">Try asking</div>')
        with gr.Row(elem_classes=["chips-wrap"]):
            ex1 = gr.Button("Compression socks \u2014 what category?", elem_classes=["chip"], size="sm")
            ex2 = gr.Button("What\u2019s under vision?", elem_classes=["chip"], size="sm")
            ex3 = gr.Button("Diabetes \u2014 what\u2019s covered?", elem_classes=["chip"], size="sm")
        with gr.Row(elem_classes=["chips-wrap"]):
            ex4 = gr.Button("Custom vs non-custom brace?", elem_classes=["chip"], size="sm")
            ex5 = gr.Button("CPAP mask vs supplies?", elem_classes=["chip"], size="sm")
            ex6 = gr.Button("Is massage therapy covered?", elem_classes=["chip"], size="sm")

    # Chat
    with gr.Column(visible=False) as chat_section:
        with gr.Row():
            gr.HTML('<span class="chat-header"><span>GreenShield ClaimBot</span></span>')
            clear_btn = gr.Button("New Chat", elem_classes=["new-chat-btn"], scale=0, min_width=90)
        chatbot = gr.Chatbot(height=420, show_label=False)
        with gr.Row():
            chat_input = gr.Textbox(
                placeholder="Ask a follow-up question...",
                show_label=False, scale=8, container=False, elem_classes=["chat-bar"],
            )
            send_btn = gr.Button("", variant="primary", scale=1, min_width=50,
                              elem_classes=["send-chat"])

    # Events
    welcome_input.submit(first_message, [welcome_input, chatbot],
                         [welcome_section, chat_section, chatbot, welcome_input])
    welcome_send.click(first_message, [welcome_input, chatbot],
                       [welcome_section, chat_section, chatbot, welcome_input])
    for btn in [ex1, ex2, ex3, ex4, ex5, ex6]:
        btn.click(first_example, [btn, chatbot],
                  [welcome_section, chat_section, chatbot, welcome_input])
    chat_input.submit(chat_message, [chat_input, chatbot], [chat_input, chatbot])
    send_btn.click(chat_message, [chat_input, chatbot], [chat_input, chatbot])
    clear_btn.click(reset_chat,
                    outputs=[welcome_section, chat_section, chatbot, welcome_input, chat_input])

demo.launch(share=False)